# RQ4 – XGBoost: Python vs SparkXGBClassifier

Compares Python XGBoost with SparkXGBClassifier for the selected configuration (XGB_04). Both branches receive identical encoded features and sample weights within each fold; preprocessing is fitted on the training portion and shared between branches.

1. Environment and file checks
2. Spark smoke test on synthetic data
3. Reconstruction of the shared train/test split and five folds
4. Check that values, labels and weights are unchanged after transfer to Spark
5. Five-fold cross-validation for both branches and out-of-fold threshold selection
6. Final training and evaluation on the held-out test lenders
7. Timing by stage

Spark runs in local mode on a single Colab machine; timings are single runs, not medians of repeated benchmarks.

In [ ]:
# Install the exact versions used in the experiment
%pip -q install pyspark==4.0.5 xgboost==3.4.1 scikit-learn==1.6.1

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import sys
import os
import subprocess

import pandas as pd
import numpy as np
import sklearn
import pyspark
import xgboost
import psutil

from xgboost.spark import SparkXGBClassifier
from IPython.display import display

BASE = Path("/content/drive/MyDrive/hmda")
SETUP_DIR = BASE / "rq4_setup"

GB_DIR = Path(
    "/content/drive/MyDrive/HMDA_Thuy/rq4_gradient_boosting"
)

BENCH_OUT = Path(
    "/content/drive/MyDrive/HMDA_Thuy/xgboost_spark_benchmark"
)

# Check the input files
required_files = {
    "Shared modeling configuration":
        SETUP_DIR / "master_modeling_config.json",
    "GB preprocessing template":
        SETUP_DIR / "gb_preprocessor.joblib",
    "Lender train/test split":
        SETUP_DIR / "lender_train_test_split.csv",
    "Saved train/test indices":
        SETUP_DIR / "train_test_indices.npz",
    "Selected XGBoost configuration":
        GB_DIR / "10_xgb_best_config.json",
    "Selected threshold":
        GB_DIR / "11_xgb_threshold.json",
}

file_checks = pd.DataFrame([
    {
        "file": name,
        "exists": path.is_file(),
    }
    for name, path in required_files.items()
])

print("TABLE 1 — FILE CHECK")
display(file_checks)

assert file_checks["exists"].all(), \
    "Some files were not found. Send the check table for review."

with open(
    SETUP_DIR / "master_modeling_config.json",
    encoding="utf-8",
) as f:
    shared_config = json.load(f)

with open(
    GB_DIR / "10_xgb_best_config.json",
    encoding="utf-8",
) as f:
    selected_xgb_config = json.load(f)

with open(
    GB_DIR / "11_xgb_threshold.json",
    encoding="utf-8",
) as f:
    selected_threshold_config = json.load(f)

# Check the data source
DATA_PATH = Path(shared_config["data_path"])
assert DATA_PATH.exists(), f"Data not found: {DATA_PATH}"

# Versions must be compatible to load the preprocessor
expected_sklearn = shared_config[
    "software_versions"
]["scikit_learn"]

assert sklearn.__version__ == expected_sklearn, (
    f"scikit-learn is {sklearn.__version__}, "
    f"required: {expected_sklearn}. "
    "If libraries were just reinstalled, restart the session and rerun this cell."
)

assert xgboost.__version__ == selected_xgb_config[
    "xgboost_version"
], "XGBoost version does not match the trained model."

ram = psutil.virtual_memory()

cpu_available = (
    len(os.sched_getaffinity(0))
    if hasattr(os, "sched_getaffinity")
    else (os.cpu_count() or 1)
)

environment_info = {
    "python": sys.version.split()[0],
    "pyspark": pyspark.__version__,
    "xgboost": xgboost.__version__,
    "scikit_learn": sklearn.__version__,
    "cpu_available": cpu_available,
    "ram_total_GB": ram.total / 1024**3,
    "ram_available_GB": ram.available / 1024**3,
}

print("\nTABLE 2 — ENVIRONMENT")
display(pd.DataFrame(
    environment_info.items(),
    columns=["item", "value"],
))

print("\nJAVA")
java_check = subprocess.run(
    ["java", "-version"],
    capture_output=True,
    text=True,
)
print(java_check.stderr or java_check.stdout)
assert java_check.returncode == 0, "Java is not working."

BENCH_OUT.mkdir(parents=True, exist_ok=True)

file_checks.to_csv(
    BENCH_OUT / "00_file_checks.csv",
    index=False,
)

with open(
    BENCH_OUT / "01_environment.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(environment_info, f, indent=2)

print("Configuration to use:", selected_xgb_config["config_id"])
print(
    "Threshold kept:",
    selected_threshold_config["optimal_threshold"],
)
print("\nPASS — environment and files are ready.")
print("No large data read and no model trained yet.")

Mounted at /content/drive
TABLE 1 — FILE CHECK


,file,exists
0,Shared modeling configuration,True
1,GB preprocessing template,True
2,Lender train/test split,True
3,Saved train/test indices,True
4,Selected XGBoost configuration,True
5,Selected threshold,True



TABLE 2 — ENVIRONMENT


,item,value
0,python,3.13.18
1,pyspark,4.0.5
2,xgboost,3.4.1
3,scikit_learn,1.6.1
4,cpu_available,2
5,ram_total_GB,12.671406
6,ram_available_GB,11.351952



JAVA
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)

Configuration to use: XGB_04
Threshold kept: 0.570807695388794

PASS — environment and files are ready.
No large data read and no model trained yet.


In [ ]:
import os
import sys
import time
import json
import gc
import numpy as np
import pandas as pd

from pyspark.sql import SparkSession, functions as F
from pyspark.ml.functions import vector_to_array
from xgboost.spark import SparkXGBClassifier
from IPython.display import display

# 1. Use the same Python for the Spark driver and workers
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

CPU_LIMIT = 2
NUM_WORKERS = 1

startup_start = time.perf_counter()

spark = (
    SparkSession.builder
    .master(f"local[{CPU_LIMIT}]")
    .appName("HMDA_XGBoost_Spark_Benchmark")
    .config("spark.driver.memory", "3g")
    .config("spark.driver.maxResultSize", "1g")
    .config("spark.task.cpus", str(CPU_LIMIT))
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.sql.execution.arrow.maxRecordsPerBatch", "10000")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

# Force Spark to actually execute a task
assert spark.range(10).count() == 10
spark_startup_seconds = time.perf_counter() - startup_start

# Verify that Spark is not reusing a session with a different configuration
assert spark.sparkContext.master == f"local[{CPU_LIMIT}]"
assert spark.sparkContext.getConf().get("spark.task.cpus") == "2"

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print("XGBoost workers:", NUM_WORKERS)
print(f"Spark startup: {spark_startup_seconds:.2f} s")

# 2. Create small synthetic data; HMDA data is not used
rng = np.random.default_rng(42)
n = 1000

x0 = rng.normal(size=n)
x1 = rng.normal(size=n)
x2 = rng.integers(0, 2, size=n).astype(float)

label = (
    x0 + 0.5 * x1 + 0.8 * x2 + rng.normal(size=n) > 0.5
).astype(float)

smoke_pdf = pd.DataFrame({
    "id": np.arange(n, dtype=np.int64),
    "x0": x0,
    "x1": x1,
    "x2": x2,
    "label": label,
})

smoke_sdf = spark.createDataFrame(smoke_pdf).repartition(1)

# 3. Test training through Spark
# This is only a technical check, not the benchmark configuration.
smoke_estimator = SparkXGBClassifier(
    features_col=["x0", "x1", "x2"],
    label_col="label",
    num_workers=NUM_WORKERS,
    device="cpu",
    tree_method="hist",
    n_estimators=10,
    max_depth=2,
    learning_rate=0.1,
    max_bin=128,
    random_state=42,
    eval_metric="logloss",
    enable_sparse_data_optim=False,
    verbosity=0,
)

print("\nChecking SparkXGBClassifier...", flush=True)

smoke_start = time.perf_counter()
smoke_spark_model = smoke_estimator.fit(smoke_sdf)

# transform is lazy: collect forces the prediction to run
smoke_predictions = (
    smoke_spark_model.transform(smoke_sdf)
    .select(
        "id",
        "label",
        vector_to_array("probability")[1].alias("probability"),
    )
    .orderBy("id")
    .toPandas()
)

smoke_seconds = time.perf_counter() - smoke_start

assert len(smoke_predictions) == n
assert smoke_predictions["id"].is_unique
assert np.array_equal(
    smoke_predictions["id"].to_numpy(),
    smoke_pdf["id"].to_numpy(),
)
assert np.isfinite(smoke_predictions["probability"]).all()
assert smoke_predictions["probability"].between(0, 1).all()

print("\nSAMPLE OUTPUT")
display(smoke_predictions.head())

# 4. Save the environment check information
spark_check = {
    "spark_version": spark.version,
    "xgboost_version": xgboost.__version__,
    "spark_master": spark.sparkContext.master,
    "spark_driver_memory": spark.sparkContext.getConf().get(
        "spark.driver.memory"
    ),
    "spark_task_cpus": spark.sparkContext.getConf().get(
        "spark.task.cpus"
    ),
    "xgboost_num_workers": NUM_WORKERS,
    "spark_startup_seconds": spark_startup_seconds,
    "synthetic_smoke_seconds": smoke_seconds,
    "synthetic_rows": n,
    "status": "PASS",
    "hmda_data_used": False,
}

with open(
    BENCH_OUT / "02_spark_startup_check.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(spark_check, f, indent=2)

# Clear the synthetic data and keep Spark running
del smoke_sdf, smoke_pdf, smoke_spark_model
del smoke_predictions, smoke_estimator
gc.collect()

print(f"\nTest training and prediction time: {smoke_seconds:.2f} s")
print("PASS — SparkXGBClassifier trained and predicted successfully.")
print("HMDA data not read yet; test results are not part of the benchmark.")

Spark version: 4.0.5
Spark master: local[2]
XGBoost workers: 1
Spark startup: 29.80 s

Checking SparkXGBClassifier...


INFO:XGBoost-PySpark:Running xgboost-3.4.1 on 1 workers with
	booster params: {'objective': 'binary:logistic', 'device': 'cpu', 'eval_metric': 'logloss', 'learning_rate': 0.1, 'max_bin': 128, 'max_depth': 2, 'random_state': 42, 'tree_method': 'hist', 'verbosity': 0, 'nthread': 2}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 10}
	dmatrix_kwargs: {'nthread': 2, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!



SAMPLE OUTPUT


,id,label,probability
0,0,1.0,0.634938
1,1,0.0,0.238649
2,2,0.0,0.481050
3,3,1.0,0.640788
4,4,0.0,0.294255



Test training and prediction time: 14.40 s
PASS — SparkXGBClassifier trained and predicted successfully.
HMDA data not read yet; test results are not part of the benchmark.


In [ ]:
import gc
import time
import json
import joblib
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

# 1. Read the shared setup
feature_config = shared_config["configurations"]["feature_selection"]
artifacts = shared_config["shared_artifacts"]
population = shared_config["population"]

FEATURES = feature_config["features"]
NUMERIC_FEATURES = feature_config["numeric_features"]
CATEGORICAL_FEATURES = feature_config["categorical_features"]

TARGET = feature_config["target"]
GROUP = feature_config["group"]
N_SPLITS = feature_config["n_splits"]
RANDOM_STATE = shared_config["random_state"]

assert len(FEATURES) == 12
assert N_SPLITS == 5

# Also check the two fold files needed for the full workflow
for key in ["cv_fold_assignment", "cv_fold_array"]:
    path = SETUP_DIR / artifacts[key]
    assert path.is_file(), f"Not found: {path}"

# 2. Read the data as in the earlier model notebook
print("Reading S1 data...", flush=True)
load_start = time.perf_counter()

table = pq.read_table(
    str(DATA_PATH),
    columns=FEATURES + [TARGET, GROUP],
)

for i, field in enumerate(table.schema):
    if pa.types.is_dictionary(field.type):
        table = table.set_column(
            i,
            field.name,
            table.column(i).cast(field.type.value_type),
        )

model_df = table.to_pandas(strings_to_categorical=True)

del table
gc.collect()

for col in NUMERIC_FEATURES:
    model_df[col] = model_df[col].astype("float32")

for col in CATEGORICAL_FEATURES + [GROUP]:
    model_df[col] = model_df[col].astype("category")

assert model_df[TARGET].isin([0, 1]).all()
assert model_df[GROUP].notna().all()

model_df[TARGET] = model_df[TARGET].astype("int8")

assert len(model_df) == population["total_applications"]
assert not np.isinf(
    model_df[NUMERIC_FEATURES].to_numpy()
).any()

shared_loading_seconds = time.perf_counter() - load_start

# 3. Reconstruct train/test by lender
print("Reconstructing train/test and the five folds...", flush=True)
split_start = time.perf_counter()

lender_split = pd.read_csv(
    SETUP_DIR / artifacts["lender_split"],
    dtype={GROUP: str},
)

assert lender_split[GROUP].is_unique

partition_assignment = model_df[GROUP].map(
    lender_split.set_index(GROUP)["partition"]
)

assert partition_assignment.notna().all()
assert partition_assignment.isin(["train", "test"]).all()

train_idx = np.flatnonzero(
    partition_assignment.eq("train").to_numpy()
)
test_idx = np.flatnonzero(
    partition_assignment.eq("test").to_numpy()
)

with np.load(
    SETUP_DIR / artifacts["train_test_indices"]
) as saved:
    assert np.array_equal(train_idx, saved["train_idx"]), (
        "Training row order differs from Hung's setup."
    )
    assert np.array_equal(test_idx, saved["test_idx"]), (
        "Test row order differs from Hung's setup."
    )

groups_train = (
    model_df[GROUP].iloc[train_idx].reset_index(drop=True)
)
groups_test = (
    model_df[GROUP].iloc[test_idx].reset_index(drop=True)
)

assert set(groups_train).isdisjoint(set(groups_test))

# 4. Reconstruct folds from the lender lists
cv_lenders = pd.read_csv(
    SETUP_DIR / artifacts["cv_fold_assignment"],
    dtype={GROUP: str},
)

assert cv_lenders[GROUP].is_unique

mapped_folds = groups_train.map(
    cv_lenders.set_index(GROUP)["validation_fold"]
)

assert mapped_folds.notna().all()
fold_assignment = mapped_folds.to_numpy(dtype=np.int8)

saved_folds = np.load(
    SETUP_DIR / artifacts["cv_fold_array"]
)

assert np.array_equal(fold_assignment, saved_folds), (
    "Folds or row order differ from Hung's setup."
)
assert set(np.unique(fold_assignment)) == set(range(1, 6))

# 5. Create the common input for both branches
X_train = (
    model_df.iloc[train_idx][FEATURES].reset_index(drop=True)
)
X_test = (
    model_df.iloc[test_idx][FEATURES].reset_index(drop=True)
)

y_train = model_df[TARGET].to_numpy()[train_idx]
y_test = model_df[TARGET].to_numpy()[test_idx]

assert len(X_train) == population["training_applications"]
assert len(X_test) == population["test_applications"]

assert set(np.unique(y_train)) == {0, 1}
assert set(np.unique(y_test)) == {0, 1}

# Load the unfitted preprocessor.
# Each fold later uses its own clone, fitted separately.
gb_preprocessor = joblib.load(
    SETUP_DIR / artifacts["gb_preprocessor"]
)

shared_partition_seconds = time.perf_counter() - split_start

# 6. Display the checks
partition_summary = pd.DataFrame([
    {
        "partition": name,
        "applications": len(labels),
        "lenders": groups.nunique(),
        "denied": int(labels.sum()),
        "denial_rate_pct": labels.mean() * 100,
    }
    for name, labels, groups in [
        ("Train", y_train, groups_train),
        ("Test", y_test, groups_test),
    ]
])

fold_rows = []

for fold in range(1, N_SPLITS + 1):
    fit_mask = fold_assignment != fold
    val_mask = fold_assignment == fold

    overlap = len(
        set(groups_train[fit_mask])
        & set(groups_train[val_mask])
    )

    assert overlap == 0
    assert set(np.unique(y_train[fit_mask])) == {0, 1}
    assert set(np.unique(y_train[val_mask])) == {0, 1}

    fold_rows.append({
        "fold": fold,
        "fit_applications": int(fit_mask.sum()),
        "validation_applications": int(val_mask.sum()),
        "validation_denial_rate_pct":
            y_train[val_mask].mean() * 100,
        "lender_overlap": overlap,
    })

fold_summary = pd.DataFrame(fold_rows)

print("\nTABLE 1 — SHARED TRAIN AND TEST SETS")
display(partition_summary.round(4))

print("\nTABLE 2 — SHARED FIVE FOLDS")
display(fold_summary.round(4))

# 7. Save the check results and the shared preparation cost
partition_summary.to_csv(
    BENCH_OUT / "03_shared_partitions.csv",
    index=False,
)

fold_summary.to_csv(
    BENCH_OUT / "04_shared_folds.csv",
    index=False,
)

with open(
    BENCH_OUT / "05_shared_preparation.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump({
        "data_source": str(DATA_PATH),
        "shared_loading_seconds": shared_loading_seconds,
        "shared_partition_seconds": shared_partition_seconds,
        "note": (
            "Shared input preparation; not an independent "
            "Pandas-versus-Spark data-loading benchmark."
        ),
    }, f, indent=2)

# Free intermediate tables, keep X_train and X_test
del model_df, partition_assignment, mapped_folds, saved_folds
gc.collect()

print(f"\nData loading time: {shared_loading_seconds:.1f} s")
print("PASS — data, row order and the five folds match Hung's setup.")
print("No HMDA model trained yet.")

Reading S1 data...
Reconstructing train/test and the five folds...

TABLE 1 — SHARED TRAIN AND TEST SETS


,partition,applications,lenders,denied,denial_rate_pct
0,Train,6308273,3927,1609889,25.5203
1,Test,1992274,982,493573,24.7744



TABLE 2 — SHARED FIVE FOLDS


,fold,fit_applications,validation_applications,validation_denial_rate_pct,lender_overlap
0,1,5046618,1261655,22.1451,0
1,2,5046618,1261655,30.5966,0
2,3,5046618,1261655,24.0303,0
3,4,5046619,1261654,23.8347,0
4,5,5046619,1261654,26.9946,0



Data loading time: 57.7 s
PASS — data, row order and the five folds match Hung's setup.
No HMDA model trained yet.


In [ ]:
import gc
import time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from pathlib import Path
from scipy import sparse
from sklearn.base import clone
from IPython.display import display

# Intermediate files are stored on the Colab disk, not on Drive
LOCAL_BENCH = Path("/content/hmda_xgb_benchmark")
LOCAL_BENCH.mkdir(parents=True, exist_ok=True)

# 1. Function converting preprocessing output to float32,
# keeping explicit zero values
def dense_float32(matrix):
    if sparse.issparse(matrix):
        result = matrix.astype(np.float32).toarray()
    else:
        result = np.asarray(matrix, dtype=np.float32)

    result = np.ascontiguousarray(result)

    assert np.isfinite(result).all()
    return result

# 2. Function computing weights from the training portion
def balanced_sample_weights(labels):
    labels = np.asarray(labels, dtype=np.int8)
    counts = np.bincount(labels, minlength=2)

    assert (counts > 0).all()

    return (
        len(labels) / (2.0 * counts)
    )[labels].astype(np.float32)

# 3. Function writing numeric input to Parquet in batches
# row_idx is kept to verify applications when Spark changes the order.
def write_encoded_parquet(
    matrix, labels, weights, row_ids, path,
    batch_size=50_000,
):
    n, d = matrix.shape

    assert len(labels) == len(weights) == len(row_ids) == n
    assert matrix.dtype == np.float32

    feature_cols = [f"x{i:03d}" for i in range(d)]

    schema = pa.schema(
        [
            ("row_idx", pa.int64()),
            ("label", pa.float32()),
            ("sample_weight", pa.float32()),
        ]
        + [(col, pa.float32()) for col in feature_cols]
    )

    with pq.ParquetWriter(
        str(path), schema, compression="snappy"
    ) as writer:
        for lo in range(0, n, batch_size):
            hi = min(lo + batch_size, n)

            arrays = [
                pa.array(row_ids[lo:hi], type=pa.int64()),
                pa.array(labels[lo:hi], type=pa.float32()),
                pa.array(weights[lo:hi], type=pa.float32()),
            ]

            arrays.extend(
                pa.array(matrix[lo:hi, j], type=pa.float32())
                for j in range(d)
            )

            writer.write_table(
                pa.Table.from_arrays(arrays, schema=schema)
            )

    return feature_cols

# 4. Sample from fold 1; the test set is not used
rng = np.random.default_rng(RANDOM_STATE)

fit_pool = np.flatnonzero(fold_assignment != 1)
val_pool = np.flatnonzero(fold_assignment == 1)

transfer_fit_idx = np.sort(
    rng.choice(
        fit_pool,
        size=min(10_000, len(fit_pool)),
        replace=False,
    )
)

transfer_val_idx = np.sort(
    rng.choice(
        val_pool,
        size=min(1_000, len(val_pool)),
        replace=False,
    )
)

assert set(groups_train.iloc[transfer_fit_idx]).isdisjoint(
    set(groups_train.iloc[transfer_val_idx])
)

# Fit preprocessing on the fit portion and apply it to validation
print("Preparing the check sample...", flush=True)

transfer_preprocessor = clone(gb_preprocessor)

transfer_fit_matrix = dense_float32(
    transfer_preprocessor.fit_transform(
        X_train.iloc[transfer_fit_idx]
    )
)

transfer_val_matrix = dense_float32(
    transfer_preprocessor.transform(
        X_train.iloc[transfer_val_idx]
    )
)

transfer_fit_y = y_train[transfer_fit_idx]
transfer_fit_weights = balanced_sample_weights(transfer_fit_y)
transfer_row_ids = train_idx[transfer_fit_idx]

# 5. Write Parquet and read it back with Spark
print("Checking the data transfer to Spark...", flush=True)

transfer_path = LOCAL_BENCH / "transfer_smoke.parquet"
transfer_start = time.perf_counter()

transfer_feature_cols = write_encoded_parquet(
    matrix=transfer_fit_matrix,
    labels=transfer_fit_y,
    weights=transfer_fit_weights,
    row_ids=transfer_row_ids,
    path=transfer_path,
)

transfer_sdf = spark.read.parquet(str(transfer_path))

# Only a small sample is collected to Python for checking.
# In the full run, the whole matrix is not collected like this.
roundtrip = (
    transfer_sdf
    .orderBy("row_idx")
    .toPandas()
)

transfer_seconds = time.perf_counter() - transfer_start

# 6. Compare values after passing through Spark
checks = {
    "Same number of rows":
        len(roundtrip) == len(transfer_fit_y),

    "Same application IDs":
        np.array_equal(
            roundtrip["row_idx"].to_numpy(),
            transfer_row_ids,
        ),

    "Same labels":
        np.array_equal(
            roundtrip["label"].to_numpy(),
            transfer_fit_y,
        ),

    "Same sample weights":
        np.array_equal(
            roundtrip["sample_weight"].to_numpy(
                dtype=np.float32
            ),
            transfer_fit_weights,
        ),

    "Same encoded feature values":
        np.array_equal(
            roundtrip[transfer_feature_cols].to_numpy(
                dtype=np.float32
            ),
            transfer_fit_matrix,
        ),
}

transfer_check = pd.DataFrame(
    checks.items(),
    columns=["check", "match"],
)

print("\nINPUT CHECK PYTHON – SPARK")
display(transfer_check)

transfer_check.to_csv(
    BENCH_OUT / "06_transfer_smoke_check.csv",
    index=False,
)

assert transfer_check["match"].all(), (
    "Some data changed when transferred to Spark. "
    "Send the check table for review."
)

print(
    f"\nEncoded columns in the sample: "
    f"{transfer_fit_matrix.shape[1]}"
)
print(f"Data transfer check time: {transfer_seconds:.2f} s")
print("PASS — Spark received the correct values, labels and weights.")

# 7. Clear the test data, keep the functions for the benchmark
del transfer_sdf, roundtrip, transfer_preprocessor
del transfer_fit_matrix, transfer_val_matrix
del transfer_fit_y, transfer_fit_weights, transfer_row_ids
del transfer_fit_idx, transfer_val_idx, fit_pool, val_pool

transfer_path.unlink()
gc.collect()

print("Test data cleared. No HMDA model trained yet.")

Preparing the check sample...
Checking the data transfer to Spark...

INPUT CHECK PYTHON – SPARK


,check,match
0,Same number of rows,True
1,Same application IDs,True
2,Same labels,True
3,Same sample weights,True
4,Same encoded feature values,True



Encoded columns in the sample: 86
Data transfer check time: 7.37 s
PASS — Spark received the correct values, labels and weights.
Test data cleared. No HMDA model trained yet.


In [ ]:
# Disable automatic partition changes by Spark during execution
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "false")

spark.catalog.clearCache()

print("Spark configuration updated.")
print("Python fold 1 checkpoint present:",
      (CV_BENCH_DIR / "Python_fold_1.npz").exists())

Spark configuration updated.
Python fold 1 checkpoint present: True


In [ ]:
import gc
import os
import json
import time
import hashlib

import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.base import clone
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
)
from threadpoolctl import threadpool_limits
from pyspark import StorageLevel
from pyspark.ml.functions import vector_to_array
from xgboost.spark import SparkXGBClassifier
from IPython.display import display

# ============================================================
# 1. Shared configuration and checkpoints
# ============================================================
CV_BENCH_DIR = BENCH_OUT / "cv_checkpoints"
CV_BENCH_DIR.mkdir(parents=True, exist_ok=True)

PYTHON_PARAMS = dict(selected_xgb_config["parameters"])
PYTHON_PARAMS["n_jobs"] = CPU_LIMIT

assert selected_xgb_config["config_id"] == "XGB_04"
assert selected_xgb_config["weight_strategy"] == "Balanced"
assert PYTHON_PARAMS["n_estimators"] == 200
assert PYTHON_PARAMS["max_depth"] == 5

# Spark sets the classifier objective and thread count from task.cpus
SPARK_PARAMS = {
    key: value
    for key, value in PYTHON_PARAMS.items()
    if key not in {
        "n_jobs", "nthread", "objective", "device",
        "enable_sparse_data_optim"
    }
}

signature = hashlib.sha256()
signature.update(
    pd.util.hash_pandas_object(
        X_train, index=True
    ).to_numpy().tobytes()
)

for array in [train_idx, y_train, fold_assignment]:
    signature.update(np.ascontiguousarray(array).tobytes())

signature.update(
    (SETUP_DIR / artifacts["gb_preprocessor"]).read_bytes()
)

benchmark_plan = {
    "config_id": "XGB_04",
    "python_parameters": PYTHON_PARAMS,
    "spark_parameters": SPARK_PARAMS,
    "spark_workers": NUM_WORKERS,
    "cpu_limit": CPU_LIMIT,
    "xgboost_version": xgb.__version__,
    "spark_version": spark.version,
    "data_signature": signature.hexdigest(),
    "folds": N_SPLITS,
    "preprocessing": "Shared within each fold; fitted on fit rows only",
    "execution_order": "Python then Spark within each fold",
    "repetitions_per_fold_per_engine": 1,
    "test_used": False,
}

plan_path = CV_BENCH_DIR / "benchmark_plan.json"

if plan_path.exists():
    with open(plan_path, encoding="utf-8") as f:
        assert json.load(f) == benchmark_plan, (
            "Setup differs from the saved checkpoint. "
            "Stop and send this message for review."
        )
else:
    with open(plan_path, "w", encoding="utf-8") as f:
        json.dump(benchmark_plan, f, indent=2)

# ============================================================
# 2. Functions for metrics and saving/loading checkpoints
# ============================================================
def benchmark_metrics(labels, probabilities):
    assert np.isfinite(probabilities).all()
    assert ((probabilities >= 0) & (probabilities <= 1)).all()

    predictions = (probabilities >= 0.5).astype("int8")

    return {
        "roc_auc": float(roc_auc_score(labels, probabilities)),
        "average_precision": float(
            average_precision_score(labels, probabilities)
        ),
        "precision_at_0_5": float(
            precision_score(labels, predictions, zero_division=0)
        ),
        "recall_at_0_5": float(
            recall_score(labels, predictions, zero_division=0)
        ),
        "f1_at_0_5": float(
            f1_score(labels, predictions, zero_division=0)
        ),
    }

def save_cv_checkpoint(path, val_idx, probabilities, result):
    temporary = path.with_suffix(".tmp")

    with open(temporary, "wb") as f:
        np.savez_compressed(
            f,
            train_position=val_idx,
            row_idx=train_idx[val_idx],
            y_true=y_train[val_idx],
            probability=probabilities,
            result_json=np.array(json.dumps(result)),
        )

    os.replace(temporary, path)

def load_cv_checkpoint(path, engine, fold, val_idx):
    with np.load(path, allow_pickle=False) as saved:
        assert np.array_equal(saved["train_position"], val_idx)
        assert np.array_equal(saved["row_idx"], train_idx[val_idx])
        assert np.array_equal(saved["y_true"], y_train[val_idx])

        probabilities = saved["probability"]
        assert probabilities.shape == (len(val_idx),)
        assert np.isfinite(probabilities).all()
        assert ((probabilities >= 0) & (probabilities <= 1)).all()

        result = json.loads(saved["result_json"].item())

    assert result["engine"] == engine
    assert result["fold"] == fold
    return result

# ============================================================
# 3. Run the five folds in turn
# ============================================================
benchmark_cv_results = []
session_start = time.perf_counter()

for fold in range(1, N_SPLITS + 1):
    fit_idx = np.flatnonzero(fold_assignment != fold)
    val_idx = np.flatnonzero(fold_assignment == fold)

    assert set(groups_train.iloc[fit_idx]).isdisjoint(
        set(groups_train.iloc[val_idx])
    )

    paths = {
        engine: CV_BENCH_DIR / f"{engine}_fold_{fold}.npz"
        for engine in ["Python", "Spark"]
    }

    pending = []

    for engine in ["Python", "Spark"]:
        if paths[engine].exists():
            benchmark_cv_results.append(
                load_cv_checkpoint(
                    paths[engine], engine, fold, val_idx
                )
            )
            print(f"Already done: {engine} | fold {fold}", flush=True)
        else:
            pending.append(engine)

    if not pending:
        continue

    print(
        f"\nFOLD {fold}/{N_SPLITS} | "
        f"fit={len(fit_idx):,} | validation={len(val_idx):,}",
        flush=True,
    )

    # --------------------------------------------------------
    # Shared preprocessing, fitted only on the fit portion
    # --------------------------------------------------------
    print("Running shared preprocessing...", flush=True)
    prep_start = time.perf_counter()

    fold_preprocessor = clone(gb_preprocessor)

    with threadpool_limits(limits=CPU_LIMIT):
        encoded_fit = dense_float32(
            fold_preprocessor.fit_transform(X_train.iloc[fit_idx])
        )
        encoded_val = dense_float32(
            fold_preprocessor.transform(X_train.iloc[val_idx])
        )

    labels_fit = y_train[fit_idx]
    labels_val = y_train[val_idx]
    weights_fit = balanced_sample_weights(labels_fit)

    common_prep_seconds = time.perf_counter() - prep_start
    encoded_columns = encoded_fit.shape[1]

    # --------------------------------------------------------
    # A. XGBoost Python
    # --------------------------------------------------------
    if "Python" in pending:
        print("Python: training...", flush=True)

        python_model = xgb.XGBClassifier(**PYTHON_PARAMS)

        fit_start = time.perf_counter()
        with threadpool_limits(limits=CPU_LIMIT):
            python_model.fit(
                encoded_fit,
                labels_fit,
                sample_weight=weights_fit,
            )
        training_seconds = time.perf_counter() - fit_start

        prediction_start = time.perf_counter()
        python_prob = python_model.predict_proba(encoded_val)[:, 1]
        prediction_seconds = time.perf_counter() - prediction_start

        evaluation_start = time.perf_counter()
        scores = benchmark_metrics(labels_val, python_prob)
        evaluation_seconds = time.perf_counter() - evaluation_start

        result = {
            "engine": "Python",
            "fold": fold,
            "fit_applications": len(fit_idx),
            "validation_applications": len(val_idx),
            "encoded_columns": encoded_columns,
            "shared_preprocessing_seconds": common_prep_seconds,
            "spark_transfer_seconds": 0.0,
            "training_seconds": training_seconds,
            "prediction_seconds": prediction_seconds,
            "evaluation_seconds": evaluation_seconds,
            **scores,
        }

        save_cv_checkpoint(
            paths["Python"], val_idx, python_prob, result
        )

        benchmark_cv_results.append(
            load_cv_checkpoint(
                paths["Python"], "Python", fold, val_idx
            )
        )

        print(
            f"SAVED Python fold {fold} | "
            f"AUC={scores['roc_auc']:.5f} | "
            f"train={training_seconds:.1f}s",
            flush=True,
        )

        del python_model, python_prob
        gc.collect()

    # --------------------------------------------------------
    # B. Transfer the same input to Spark
    # --------------------------------------------------------
    if "Spark" in pending:
        print("Spark: transferring data...", flush=True)

        fit_path = LOCAL_BENCH / f"fit_fold_{fold}.parquet"
        val_path = LOCAL_BENCH / f"val_fold_{fold}.parquet"

        transfer_start = time.perf_counter()

        spark_feature_cols = write_encoded_parquet(
            encoded_fit,
            labels_fit,
            weights_fit,
            train_idx[fit_idx],
            fit_path,
        )

        val_feature_cols = write_encoded_parquet(
            encoded_val,
            labels_val,
            np.ones(len(val_idx), dtype=np.float32),
            train_idx[val_idx],
            val_path,
        )

        assert spark_feature_cols == val_feature_cols

        # Free the Python matrix before Spark training,
        # to avoid holding two large copies in memory.
        del encoded_fit, encoded_val, fold_preprocessor, weights_fit
        gc.collect()

        spark_fit = None
        spark_val = None

        try:
            # Cache on disk to reduce memory pressure.
            # count forces Spark to finish reading and materialising.
            spark_fit = (
                spark.read.parquet(str(fit_path))
                .repartition(NUM_WORKERS)
                .persist(StorageLevel.DISK_ONLY)
            )
            spark_val = (
                spark.read.parquet(str(val_path))
                .repartition(NUM_WORKERS)
                .persist(StorageLevel.DISK_ONLY)
            )

            assert spark_fit.count() == len(fit_idx)
            assert spark_val.count() == len(val_idx)

            transfer_seconds = time.perf_counter() - transfer_start

            spark_estimator = SparkXGBClassifier(
                **SPARK_PARAMS,
                features_col=spark_feature_cols,
                label_col="label",
                weight_col="sample_weight",
                num_workers=NUM_WORKERS,
                device="cpu",
                enable_sparse_data_optim=False,
                force_repartition=True,
            )

            print("Spark: training...", flush=True)

            fit_start = time.perf_counter()
            spark_model = spark_estimator.fit(spark_fit)
            training_seconds = time.perf_counter() - fit_start

            print("Spark: predicting validation...", flush=True)

            prediction_start = time.perf_counter()

            # Collect IDs, labels and scores together.
            # Do not assume Spark preserves row order.
            predicted = (
                spark_model.transform(spark_val)
                .select(
                    "row_idx",
                    "label",
                    vector_to_array("probability")[1].alias(
                        "probability"
                    ),
                )
                .toPandas()
            )

            prediction_seconds = time.perf_counter() - prediction_start

            # Restore the application order before assembling OOF predictions
            alignment_start = time.perf_counter()

            assert predicted["row_idx"].is_unique

            predicted = (
                predicted.set_index("row_idx")
                .reindex(train_idx[val_idx])
            )

            assert predicted["label"].notna().all()
            assert np.array_equal(
                predicted["label"].to_numpy(),
                labels_val,
            )

            spark_prob = predicted["probability"].to_numpy(
                dtype=np.float32
            )

            alignment_seconds = time.perf_counter() - alignment_start

            evaluation_start = time.perf_counter()
            scores = benchmark_metrics(labels_val, spark_prob)
            evaluation_seconds = time.perf_counter() - evaluation_start

            result = {
                "engine": "Spark",
                "fold": fold,
                "fit_applications": len(fit_idx),
                "validation_applications": len(val_idx),
                "encoded_columns": encoded_columns,
                "shared_preprocessing_seconds": common_prep_seconds,
                "spark_transfer_seconds": transfer_seconds,
                "training_seconds": training_seconds,
                "prediction_seconds": prediction_seconds,
                "alignment_seconds": alignment_seconds,
                "evaluation_seconds": evaluation_seconds,
                **scores,
            }

            save_cv_checkpoint(
                paths["Spark"], val_idx, spark_prob, result
            )

            benchmark_cv_results.append(
                load_cv_checkpoint(
                    paths["Spark"], "Spark", fold, val_idx
                )
            )

            print(
                f"SAVED Spark fold {fold} | "
                f"AUC={scores['roc_auc']:.5f} | "
                f"train={training_seconds:.1f}s | "
                f"transfer={transfer_seconds:.1f}s",
                flush=True,
            )

            del spark_model, spark_estimator, predicted, spark_prob

        finally:
            if spark_fit is not None:
                spark_fit.unpersist(blocking=True)
            if spark_val is not None:
                spark_val.unpersist(blocking=True)

            for path in [fit_path, val_path]:
                if path.exists():
                    path.unlink()

            del spark_fit, spark_val
            gc.collect()

    else:
        del encoded_fit, encoded_val, fold_preprocessor, weights_fit
        gc.collect()

    # Report progress after each fold
    pd.DataFrame(benchmark_cv_results).to_csv(
        BENCH_OUT / "07_cv_fold_results.csv",
        index=False,
        encoding="utf-8-sig",
    )

# ============================================================
# 4. Summarise the results
# ============================================================
benchmark_cv_results = pd.DataFrame(benchmark_cv_results)

assert len(benchmark_cv_results) == 2 * N_SPLITS
assert not benchmark_cv_results.duplicated(["engine", "fold"]).any()

benchmark_cv_summary = (
    benchmark_cv_results.groupby("engine", as_index=False)
    .agg(
        completed_folds=("fold", "nunique"),
        mean_cv_roc_auc=("roc_auc", "mean"),
        std_cv_roc_auc=("roc_auc", "std"),
        mean_cv_AP=("average_precision", "mean"),
        mean_cv_f1_at_0_5=("f1_at_0_5", "mean"),
        total_training_seconds=("training_seconds", "sum"),
        total_prediction_seconds=("prediction_seconds", "sum"),
        total_spark_transfer_seconds=("spark_transfer_seconds", "sum"),
    )
)

assert benchmark_cv_summary["completed_folds"].eq(5).all()

benchmark_cv_results.to_csv(
    BENCH_OUT / "07_cv_fold_results.csv",
    index=False,
    encoding="utf-8-sig",
)

benchmark_cv_summary.to_csv(
    BENCH_OUT / "08_cv_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\nFIVE-FOLD RESULTS")
display(benchmark_cv_summary.round(6))

print(
    f"\nCell run time in this session: "
    f"{(time.perf_counter() - session_start) / 60:.1f} min"
)
print("PASS — all 10 training runs complete; out-of-fold predictions saved.")
print("Test set not used yet; thresholds not yet selected for each branch.")

Already done: Python | fold 1
Already done: Spark | fold 1
Already done: Python | fold 2
Already done: Spark | fold 2
Already done: Python | fold 3
Already done: Spark | fold 3
Already done: Python | fold 4
Already done: Spark | fold 4
Already done: Python | fold 5
Already done: Spark | fold 5

FIVE-FOLD RESULTS


,engine,completed_folds,mean_cv_roc_auc,std_cv_roc_auc,mean_cv_AP,mean_cv_f1_at_0_5,total_training_seconds,total_prediction_seconds,total_spark_transfer_seconds
0,Python,5,0.857654,0.026974,0.749649,0.652729,1011.438044,27.243673,0.000000
1,Spark,5,0.857654,0.026974,0.749649,0.652729,1873.710752,217.504985,487.619961



Cell run time in this session: 0.1 min
PASS — all 10 training runs complete; out-of-fold predictions saved.
Test set not used yet; thresholds not yet selected for each branch.


In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_recall_curve,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

# 1. Assemble out-of-fold predictions at their positions in the training set
oof_predictions = {}
threshold_results = {}
metric_rows = []

for engine in ["Python", "Spark"]:
    probabilities = np.full(len(y_train), np.nan, dtype=np.float32)
    seen = np.zeros(len(y_train), dtype=np.int8)

    for fold in range(1, N_SPLITS + 1):
        checkpoint_path = (
            CV_BENCH_DIR / f"{engine}_fold_{fold}.npz"
        )

        with np.load(checkpoint_path, allow_pickle=False) as saved:
            positions = saved["train_position"].astype(np.int64)
            saved_ids = saved["row_idx"]
            saved_labels = saved["y_true"]
            fold_probabilities = saved["probability"]

        assert np.array_equal(
            positions, np.flatnonzero(fold_assignment == fold)
        ), f"{engine} fold {fold}: wrong validation positions"

        assert np.array_equal(
            saved_ids, train_idx[positions]
        ), f"{engine} fold {fold}: wrong application IDs"

        assert np.array_equal(
            saved_labels, y_train[positions]
        ), f"{engine} fold {fold}: wrong labels"

        assert np.all(seen[positions] == 0), "Duplicate out-of-fold predictions"

        probabilities[positions] = fold_probabilities
        seen[positions] += 1

    assert np.all(seen == 1), "Some applications have missing or duplicate predictions"
    assert np.isfinite(probabilities).all()
    assert ((probabilities >= 0) & (probabilities <= 1)).all()

    oof_predictions[engine] = probabilities

    # 2. Select the threshold maximising F1 on out-of-fold predictions
    precision_curve, recall_curve, thresholds = precision_recall_curve(
        y_train, probabilities
    )

    f1_curve = np.divide(
        2 * precision_curve[:-1] * recall_curve[:-1],
        precision_curve[:-1] + recall_curve[:-1],
        out=np.zeros_like(precision_curve[:-1]),
        where=(precision_curve[:-1] + recall_curve[:-1]) > 0,
    )

    best_index = int(np.argmax(f1_curve))
    selected_threshold = float(thresholds[best_index])

    threshold_results[engine] = {
        "threshold": selected_threshold,
        "selection_metric": "Maximum OOF F1",
        "selection_data": "Training-set out-of-fold predictions only",
        "oof_applications": int(len(y_train)),
        "test_used_for_selection": False,
    }

    # 3. Report at both the default and the selected threshold
    auc = roc_auc_score(y_train, probabilities)
    ap = average_precision_score(y_train, probabilities)

    for threshold_type, threshold in [
        ("Default", 0.5),
        ("Optimized", selected_threshold),
    ]:
        predicted_labels = (probabilities >= threshold).astype(np.int8)
        tn, fp, fn, tp = confusion_matrix(
            y_train, predicted_labels, labels=[0, 1]
        ).ravel()

        metric_rows.append({
            "engine": engine,
            "threshold_type": threshold_type,
            "threshold": threshold,
            "roc_auc": auc,
            "average_precision": ap,
            "precision": precision_score(
                y_train, predicted_labels, zero_division=0
            ),
            "recall": recall_score(
                y_train, predicted_labels, zero_division=0
            ),
            "f1": f1_score(
                y_train, predicted_labels, zero_division=0
            ),
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
        })

# 4. Compare probabilities on the same applications
absolute_difference = np.abs(
    oof_predictions["Python"] - oof_predictions["Spark"]
)

comparison = pd.DataFrame([{
    "applications": len(y_train),
    "exactly_equal": np.array_equal(
        oof_predictions["Python"], oof_predictions["Spark"]
    ),
    "max_abs_probability_difference": float(absolute_difference.max()),
    "mean_abs_probability_difference": float(absolute_difference.mean()),
    "share_within_1e_6_pct": float(
        100 * np.mean(absolute_difference <= 1e-6)
    ),
}])

oof_metrics = pd.DataFrame(metric_rows)

# 5. Save the results
with open(
    BENCH_OUT / "09_oof_thresholds.json", "w", encoding="utf-8"
) as file:
    json.dump(threshold_results, file, indent=2, ensure_ascii=False)

oof_metrics.to_csv(
    BENCH_OUT / "10_oof_metrics.csv", index=False
)
comparison.to_csv(
    BENCH_OUT / "11_oof_probability_comparison.csv", index=False
)

print("TABLE 1 — OUT-OF-FOLD RESULTS ON THE TRAINING SET")
display(oof_metrics)

print("\nTABLE 2 — PYTHON VS SPARK PROBABILITY COMPARISON")
display(comparison)

print("\nSELECTED THRESHOLDS")
for engine, result in threshold_results.items():
    print(f"{engine}: {result['threshold']:.9f}")

print("\nPASS — thresholds saved for both branches.")
print("Test set not used yet; final model not trained yet.")

TABLE 1 — OUT-OF-FOLD RESULTS ON THE TRAINING SET


,engine,threshold_type,threshold,roc_auc,average_precision,precision,recall,f1,tn,fp,fn,tp
0,Python,Default,0.500000,0.859236,0.753251,0.61313,0.706963,0.656712,3980251,718133,471757,1138132
1,Python,Optimized,0.570808,0.859236,0.753251,0.68949,0.641249,0.664495,4233472,464912,577549,1032340
2,Spark,Default,0.500000,0.859236,0.753251,0.61313,0.706963,0.656712,3980251,718133,471757,1138132
3,Spark,Optimized,0.570808,0.859236,0.753251,0.68949,0.641249,0.664495,4233472,464912,577549,1032340



TABLE 2 — PYTHON VS SPARK PROBABILITY COMPARISON


,applications,exactly_equal,max_abs_probability_difference,mean_abs_probability_difference,share_within_1e_6_pct
0,6308273,True,0.0,0.0,100.0



SELECTED THRESHOLDS
Python: 0.570807695
Spark: 0.570807695

PASS — thresholds saved for both branches.
Test set not used yet; final model not trained yet.


In [ ]:
import gc
import json
import time
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.base import clone
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, brier_score_loss,
)
from threadpoolctl import threadpool_limits

# Reload the thresholds fixed from OOF predictions
with open(
    BENCH_OUT / "09_oof_thresholds.json", encoding="utf-8"
) as file:
    final_thresholds = json.load(file)

# Free OOF arrays that are no longer needed
for variable_name in [
    "oof_predictions", "probabilities",
    "absolute_difference", "predicted_labels",
]:
    globals().pop(variable_name, None)
gc.collect()

# 1. Fit preprocessing ONLY on train, then transform test
print("Preprocessing the full train and test sets...")

preprocessing_start = time.perf_counter()
final_preprocessor = clone(gb_preprocessor)

with threadpool_limits(limits=CPU_LIMIT):
    final_train_matrix = dense_float32(
        final_preprocessor.fit_transform(X_train)
    )
    final_test_matrix = dense_float32(
        final_preprocessor.transform(X_test)
    )

final_preprocessing_seconds = (
    time.perf_counter() - preprocessing_start
)

assert final_train_matrix.shape[0] == len(y_train)
assert final_test_matrix.shape[0] == len(y_test)
assert final_train_matrix.shape[1] == final_test_matrix.shape[1]

final_weights = balanced_sample_weights(y_train)

joblib.dump(
    final_preprocessor,
    BENCH_OUT / "12_final_preprocessor.joblib"
)

print(f"Train: {final_train_matrix.shape}")
print(f"Test:  {final_test_matrix.shape}")

# 2. Prepare the same encoded data for the Spark branch
# This is the Spark data-transfer cost, recorded separately.
print("\nWriting temporary data for Spark...")

final_fit_path = LOCAL_BENCH / "final_train.parquet"
final_test_path = LOCAL_BENCH / "final_test.parquet"

staging_start = time.perf_counter()

final_feature_cols = write_encoded_parquet(
    final_train_matrix,
    y_train,
    final_weights,
    train_idx,
    final_fit_path,
)

test_feature_cols = write_encoded_parquet(
    final_test_matrix,
    y_test,
    np.ones(len(y_test), dtype=np.float32),
    test_idx,
    final_test_path,
)

assert final_feature_cols == test_feature_cols

final_parquet_staging_seconds = (
    time.perf_counter() - staging_start
)

# 3. Train Python with the fixed configuration
print("\nPython: training on the full training set...")

final_python_model = xgb.XGBClassifier(**PYTHON_PARAMS)

training_start = time.perf_counter()
with threadpool_limits(limits=CPU_LIMIT):
    final_python_model.fit(
        final_train_matrix,
        y_train,
        sample_weight=final_weights,
    )
python_final_training_seconds = (
    time.perf_counter() - training_start
)

final_python_model.save_model(
    str(BENCH_OUT / "13_python_final_model.ubj")
)

# 4. Predict the test set in batches to save memory
print("Python: predicting test...")

prediction_start = time.perf_counter()
python_test_probability = np.empty(
    len(y_test), dtype=np.float32
)

batch_size = 100_000
with threadpool_limits(limits=CPU_LIMIT):
    for lo in range(0, len(y_test), batch_size):
        hi = min(lo + batch_size, len(y_test))
        python_test_probability[lo:hi] = (
            final_python_model.predict_proba(
                final_test_matrix[lo:hi]
            )[:, 1]
        )

python_final_prediction_seconds = (
    time.perf_counter() - prediction_start
)

assert np.isfinite(python_test_probability).all()
assert (
    (python_test_probability >= 0)
    & (python_test_probability <= 1)
).all()

# 5. Evaluate at the default and the fixed OOF threshold
evaluation_start = time.perf_counter()

python_test_auc = roc_auc_score(
    y_test, python_test_probability
)
python_test_ap = average_precision_score(
    y_test, python_test_probability
)
python_test_brier = brier_score_loss(
    y_test, python_test_probability
)

python_final_rows = []

for threshold_type, threshold in [
    ("Default", 0.5),
    ("Optimized", final_thresholds["Python"]["threshold"]),
]:
    predicted = (
        python_test_probability >= threshold
    ).astype(np.int8)

    tn, fp, fn, tp = confusion_matrix(
        y_test, predicted, labels=[0, 1]
    ).ravel()

    python_final_rows.append({
        "engine": "Python",
        "threshold_type": threshold_type,
        "threshold": float(threshold),
        "roc_auc": python_test_auc,
        "average_precision": python_test_ap,
        "accuracy": accuracy_score(y_test, predicted),
        "precision": precision_score(
            y_test, predicted, zero_division=0
        ),
        "recall": recall_score(
            y_test, predicted, zero_division=0
        ),
        "f1": f1_score(
            y_test, predicted, zero_division=0
        ),
        "specificity": float(tn / (tn + fp)),
        "brier_score": python_test_brier,
        "tn": int(tn), "fp": int(fp),
        "fn": int(fn), "tp": int(tp),
    })

python_final_evaluation_seconds = (
    time.perf_counter() - evaluation_start
)

# 6. Save results before running Spark
python_final_metrics = pd.DataFrame(python_final_rows)
python_final_metrics.to_csv(
    BENCH_OUT / "14_python_test_metrics.csv", index=False
)

np.savez_compressed(
    BENCH_OUT / "15_python_test_predictions.npz",
    row_idx=test_idx,
    y_true=y_test,
    probability=python_test_probability,
)

final_run_metadata = {
    "config_id": selected_xgb_config["config_id"],
    "train_applications": int(len(y_train)),
    "test_applications": int(len(y_test)),
    "encoded_columns": int(final_train_matrix.shape[1]),
    "feature_columns": final_feature_cols,
    "shared_preprocessing_seconds": final_preprocessing_seconds,
    "spark_parquet_staging_seconds": final_parquet_staging_seconds,
    "python_training_seconds": python_final_training_seconds,
    "python_prediction_seconds": python_final_prediction_seconds,
    "python_evaluation_seconds": python_final_evaluation_seconds,
    "execution_order": "Python then Spark",
    "repetitions_per_engine": 1,
}

with open(
    BENCH_OUT / "16_final_run_metadata.json",
    "w", encoding="utf-8"
) as file:
    json.dump(final_run_metadata, file, indent=2)

# Free large matrices before training Spark
del final_train_matrix, final_test_matrix
del final_python_model, final_preprocessor, final_weights
gc.collect()

print("\nPYTHON RESULTS — TEST ON UNSEEN LENDERS")
display(python_final_metrics)

print(
    f"\nPython training time: "
    f"{python_final_training_seconds:.1f} s"
)
print(
    f"Python prediction time: "
    f"{python_final_prediction_seconds:.1f} s"
)

print("\nPASS — Python model and test results saved.")
print("Data prepared for Spark; final Spark model not trained yet.")

Preprocessing the full train and test sets...
Train: (6308273, 88)
Test:  (1992274, 88)

Writing temporary data for Spark...

Python: training on the full training set...
Python: predicting test...

PYTHON RESULTS — TEST ON UNSEEN LENDERS


,engine,threshold_type,threshold,roc_auc,average_precision,accuracy,precision,recall,f1,specificity,brier_score,tn,fp,fn,tp
0,Python,Default,0.500000,0.860408,0.737704,0.810276,0.596549,0.723506,0.653922,0.838852,0.141933,1257189,241512,136470,357103
1,Python,Optimized,0.570808,0.860408,0.737704,0.830521,0.654309,0.669765,0.661947,0.883463,0.141933,1324047,174654,162995,330578



Python training time: 256.1 s
Python prediction time: 10.8 s

PASS — Python model and test results saved.
Data prepared for Spark; final Spark model not trained yet.


In [ ]:
import gc
import json
import time
import numpy as np
import pandas as pd

from pyspark import StorageLevel
from pyspark.ml.functions import vector_to_array
from xgboost.spark import SparkXGBClassifier

# 1. Read the saved configuration and results
with open(
    BENCH_OUT / "16_final_run_metadata.json", encoding="utf-8"
) as file:
    final_run_metadata = json.load(file)

with open(
    BENCH_OUT / "09_oof_thresholds.json", encoding="utf-8"
) as file:
    final_thresholds = json.load(file)

final_feature_cols = final_run_metadata["feature_columns"]

final_fit_path = LOCAL_BENCH / "final_train.parquet"
final_test_path = LOCAL_BENCH / "final_test.parquet"

assert final_fit_path.exists(), "Temporary Spark training data missing"
assert final_test_path.exists(), "Temporary Spark test data missing"

# Keep the partitioning that worked in CV
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set(
    "spark.sql.adaptive.coalescePartitions.enabled", "false"
)
spark.catalog.clearCache()
gc.collect()

spark_final_train = None
spark_final_test = None

try:
    # 2. Read and prepare the Spark data
    print("Spark: reading and preparing data...")

    transfer_start = time.perf_counter()

    spark_final_train = (
        spark.read.parquet(str(final_fit_path))
        .repartition(NUM_WORKERS)
        .persist(StorageLevel.DISK_ONLY)
    )

    spark_final_test = (
        spark.read.parquet(str(final_test_path))
        .repartition(NUM_WORKERS)
        .persist(StorageLevel.DISK_ONLY)
    )

    assert spark_final_train.count() == len(y_train)
    assert spark_final_test.count() == len(y_test)

    spark_read_seconds = time.perf_counter() - transfer_start

    # 3. Train on the full training set
    print(f"Spark: training on {len(y_train):,} applications...")

    spark_final_estimator = SparkXGBClassifier(
        **SPARK_PARAMS,
        features_col=final_feature_cols,
        label_col="label",
        weight_col="sample_weight",
        num_workers=NUM_WORKERS,
        device="cpu",
        enable_sparse_data_optim=False,
        force_repartition=True,
    )

    training_start = time.perf_counter()
    spark_final_model = spark_final_estimator.fit(spark_final_train)
    spark_final_training_seconds = (
        time.perf_counter() - training_start
    )

    # Save the trained booster; the shared preprocessor was saved in the previous cell
    spark_final_model.get_booster().save_model(
        str(BENCH_OUT / "17_spark_final_booster.ubj")
    )

    # 4. Predict the test set and collect the results to Python for comparison
    print("Spark: predicting test and collecting results...")

    prediction_start = time.perf_counter()

    spark_test_predictions = (
        spark_final_model.transform(spark_final_test)
        .select(
            "row_idx",
            "label",
            vector_to_array("probability")[1].alias("probability"),
        )
        .toPandas()
    )

    spark_final_prediction_seconds = (
        time.perf_counter() - prediction_start
    )

    # 5. Order by the test IDs of the Python branch
    alignment_start = time.perf_counter()

    assert len(spark_test_predictions) == len(y_test)
    assert spark_test_predictions["row_idx"].is_unique

    spark_test_predictions = (
        spark_test_predictions
        .set_index("row_idx")
        .reindex(test_idx)
    )

    assert spark_test_predictions["label"].notna().all()
    assert np.array_equal(
        spark_test_predictions["label"].to_numpy(),
        np.asarray(y_test),
    ), "Test labels or IDs do not match"

    spark_test_probability = (
        spark_test_predictions["probability"]
        .to_numpy(dtype=np.float32)
    )

    assert np.isfinite(spark_test_probability).all()
    assert (
        (spark_test_probability >= 0)
        & (spark_test_probability <= 1)
    ).all()

    spark_alignment_seconds = (
        time.perf_counter() - alignment_start
    )

    # Save predictions before computing the comparison table
    np.savez_compressed(
        BENCH_OUT / "18_spark_test_predictions.npz",
        row_idx=test_idx,
        y_true=y_test,
        probability=spark_test_probability,
    )

    # 6. Evaluate with the threshold fixed from OOF predictions
    evaluation_start = time.perf_counter()

    spark_auc = roc_auc_score(y_test, spark_test_probability)
    spark_ap = average_precision_score(y_test, spark_test_probability)
    spark_brier = brier_score_loss(y_test, spark_test_probability)

    spark_final_rows = []

    for threshold_type, threshold in [
        ("Default", 0.5),
        ("Optimized", final_thresholds["Spark"]["threshold"]),
    ]:
        predicted = (
            spark_test_probability >= threshold
        ).astype(np.int8)

        tn, fp, fn, tp = confusion_matrix(
            y_test, predicted, labels=[0, 1]
        ).ravel()

        spark_final_rows.append({
            "engine": "Spark",
            "threshold_type": threshold_type,
            "threshold": float(threshold),
            "roc_auc": spark_auc,
            "average_precision": spark_ap,
            "accuracy": accuracy_score(y_test, predicted),
            "precision": precision_score(
                y_test, predicted, zero_division=0
            ),
            "recall": recall_score(
                y_test, predicted, zero_division=0
            ),
            "f1": f1_score(y_test, predicted, zero_division=0),
            "specificity": float(tn / (tn + fp)),
            "brier_score": spark_brier,
            "tn": int(tn), "fp": int(fp),
            "fn": int(fn), "tp": int(tp),
        })

    spark_final_evaluation_seconds = (
        time.perf_counter() - evaluation_start
    )

    spark_final_metrics = pd.DataFrame(spark_final_rows)
    spark_final_metrics.to_csv(
        BENCH_OUT / "19_spark_test_metrics.csv", index=False
    )

    # 7. Check directly against the saved Python predictions
    with np.load(
        BENCH_OUT / "15_python_test_predictions.npz",
        allow_pickle=False,
    ) as saved:
        assert np.array_equal(saved["row_idx"], test_idx)
        assert np.array_equal(saved["y_true"], y_test)
        python_probability = saved["probability"].copy()

    differences = np.abs(
        python_probability.astype(np.float64)
        - spark_test_probability.astype(np.float64)
    )

    test_probability_comparison = pd.DataFrame([{
        "applications": len(y_test),
        "same_test_ids_and_labels": True,
        "exactly_equal": np.array_equal(
            python_probability, spark_test_probability
        ),
        "max_abs_probability_difference": float(differences.max()),
        "mean_abs_probability_difference": float(differences.mean()),
        "share_within_1e_6_pct": float(
            100 * np.mean(differences <= 1e-6)
        ),
    }])

    test_probability_comparison.to_csv(
        BENCH_OUT / "20_test_probability_comparison.csv", index=False
    )

    python_metrics = pd.read_csv(
        BENCH_OUT / "14_python_test_metrics.csv"
    )
    final_model_comparison = pd.concat(
        [python_metrics, spark_final_metrics], ignore_index=True
    )
    final_model_comparison.to_csv(
        BENCH_OUT / "21_final_model_comparison.csv", index=False
    )

    # 8. Save timing by stage
    final_run_metadata.update({
        "spark_read_and_materialize_seconds": spark_read_seconds,
        "spark_total_transfer_seconds": (
            final_run_metadata["spark_parquet_staging_seconds"]
            + spark_read_seconds
        ),
        "spark_training_seconds": spark_final_training_seconds,
        "spark_prediction_and_collect_seconds": (
            spark_final_prediction_seconds
        ),
        "spark_alignment_seconds": spark_alignment_seconds,
        "spark_evaluation_seconds": spark_final_evaluation_seconds,
        "spark_workers": NUM_WORKERS,
        "spark_adaptive_execution": False,
        "spark_force_repartition": True,
    })

    with open(
        BENCH_OUT / "16_final_run_metadata.json",
        "w", encoding="utf-8",
    ) as file:
        json.dump(final_run_metadata, file, indent=2)

    print("\nTABLE 1 — TEST RESULTS OF BOTH BRANCHES")
    display(final_model_comparison)

    print("\nTABLE 2 — TEST PROBABILITY COMPARISON")
    display(test_probability_comparison)

    print(
        f"\nSpark training time: "
        f"{spark_final_training_seconds:.1f} s"
    )
    print(
        f"Spark prediction and collection time: "
        f"{spark_final_prediction_seconds:.1f} s"
    )
    print("\nPASS — Spark results and the test comparison table saved.")
    print("Threshold kept from OOF; not adjusted on the test set.")

finally:
    if spark_final_train is not None:
        spark_final_train.unpersist(blocking=True)
    if spark_final_test is not None:
        spark_final_test.unpersist(blocking=True)
    gc.collect()

Spark: reading and preparing data...
Spark: training on 6,308,273 applications...


INFO:XGBoost-PySpark:Running xgboost-3.4.1 on 1 workers with
	booster params: {'objective': 'binary:logistic', 'booster': 'gbtree', 'colsample_bytree': 1.0, 'device': 'cpu', 'eval_metric': 'logloss', 'learning_rate': 0.1, 'max_bin': 128, 'max_depth': 5, 'random_state': 42, 'reg_lambda': 1.0, 'subsample': 1.0, 'tree_method': 'hist', 'verbosity': 0, 'nthread': 2}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 200}
	dmatrix_kwargs: {'nthread': 2, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!


Spark: predicting test and collecting results...

TABLE 1 — TEST RESULTS OF BOTH BRANCHES


,engine,threshold_type,threshold,roc_auc,average_precision,accuracy,precision,recall,f1,specificity,brier_score,tn,fp,fn,tp
0,Python,Default,0.500000,0.860408,0.737704,0.810276,0.596549,0.723506,0.653922,0.838852,0.141933,1257189,241512,136470,357103
1,Python,Optimized,0.570808,0.860408,0.737704,0.830521,0.654309,0.669765,0.661947,0.883463,0.141933,1324047,174654,162995,330578
2,Spark,Default,0.500000,0.860408,0.737704,0.810276,0.596549,0.723506,0.653922,0.838852,0.141933,1257189,241512,136470,357103
3,Spark,Optimized,0.570808,0.860408,0.737704,0.830521,0.654309,0.669765,0.661947,0.883463,0.141933,1324047,174654,162995,330578



TABLE 2 — TEST PROBABILITY COMPARISON


,applications,same_test_ids_and_labels,exactly_equal,max_abs_probability_difference,mean_abs_probability_difference,share_within_1e_6_pct
0,1992274,True,True,0.0,0.0,100.0



Spark training time: 462.5 s
Spark prediction and collection time: 70.5 s

PASS — Spark results and the test comparison table saved.
Threshold kept from OOF; not adjusted on the test set.


In [ ]:
import json
import pandas as pd

# 1. Read CV and final-training times
cv_summary = pd.read_csv(
    BENCH_OUT / "08_cv_summary.csv"
).set_index("engine")

with open(
    BENCH_OUT / "16_final_run_metadata.json", encoding="utf-8"
) as file:
    timing = json.load(file)

# 2. Summarise the measured stages
timing_rows = [
    {
        "stage": "CV training — 5 folds",
        "Python_seconds": cv_summary.loc[
            "Python", "total_training_seconds"
        ],
        "Spark_seconds": cv_summary.loc[
            "Spark", "total_training_seconds"
        ],
    },
    {
        "stage": "Final training",
        "Python_seconds": timing["python_training_seconds"],
        "Spark_seconds": timing["spark_training_seconds"],
    },
    {
        "stage": "CV prediction — 5 folds",
        "Python_seconds": cv_summary.loc[
            "Python", "total_prediction_seconds"
        ],
        "Spark_seconds": cv_summary.loc[
            "Spark", "total_prediction_seconds"
        ],
    },
    {
        "stage": "Final test prediction",
        "Python_seconds": timing["python_prediction_seconds"],
        "Spark_seconds": timing["spark_prediction_and_collect_seconds"],
    },
    {
        "stage": "CV data transfer to Spark",
        "Python_seconds": 0.0,
        "Spark_seconds": cv_summary.loc[
            "Spark", "total_spark_transfer_seconds"
        ],
    },
    {
        "stage": "Final data transfer to Spark",
        "Python_seconds": 0.0,
        "Spark_seconds": timing["spark_total_transfer_seconds"],
    },
]

timing_table = pd.DataFrame(timing_rows)

subtotal = {
    "stage": "Subtotal of listed stages",
    "Python_seconds": timing_table["Python_seconds"].sum(),
    "Spark_seconds": timing_table["Spark_seconds"].sum(),
}

timing_table = pd.concat(
    [timing_table, pd.DataFrame([subtotal])],
    ignore_index=True,
)

timing_table["Python_minutes"] = (
    timing_table["Python_seconds"] / 60
)
timing_table["Spark_minutes"] = (
    timing_table["Spark_seconds"] / 60
)

timing_table.to_csv(
    BENCH_OUT / "22_timing_summary.csv", index=False
)

print("TIMING TABLE — MEASURED STAGES")
display(timing_table.round(3))

print(
    "\nShared preprocessing for the final run: "
    f"{timing['shared_preprocessing_seconds']:.1f} s."
)
print(
    "Subtotal excludes shared preprocessing, metric calculation, "
    "ID alignment and saving results."
)
print(
    "Spark prediction time includes collecting results to Python."
)
print(
    "Each fold and the final training run once per branch; "
    "these are not medians of repeated benchmark runs."
)
print("\nPASS — timing summary saved.")

TIMING TABLE — MEASURED STAGES


,stage,Python_seconds,Spark_seconds,Python_minutes,Spark_minutes
0,CV training — 5 folds,1011.438,1873.711,16.857,31.229
1,Final training,256.074,462.483,4.268,7.708
2,CV prediction — 5 folds,27.244,217.505,0.454,3.625
3,Final test prediction,10.840,70.522,0.181,1.175
4,CV data transfer to Spark,0.000,487.620,0.000,8.127
5,Final data transfer to Spark,0.000,402.475,0.000,6.708
6,Subtotal of listed stages,1305.596,3514.316,21.760,58.572



Shared preprocessing for the final run: 67.6 s.
Subtotal excludes shared preprocessing, metric calculation, ID alignment and saving results.
Spark prediction time includes collecting results to Python.
Each fold and the final training run once per branch; these are not medians of repeated benchmark runs.

PASS — timing summary saved.


In [ ]:
import pandas as pd

results = pd.read_csv(
    BENCH_OUT / "21_final_model_comparison.csv"
)

# Get test results at each branch's fixed threshold
selected = (
    results.loc[results["threshold_type"] == "Optimized"]
    .set_index("engine")
)

metrics = {
    "threshold": "Decision threshold",
    "roc_auc": "ROC-AUC",
    "average_precision": "Average Precision (AP)",
    "accuracy": "Accuracy",
    "precision": "Precision",
    "recall": "Recall",
    "f1": "F1-score",
    "specificity": "Specificity",
    "brier_score": "Brier score",
}

comparison = (
    selected.loc[["Python", "Spark"], list(metrics)]
    .T
    .rename(index=metrics)
)

comparison["Spark − Python"] = (
    comparison["Spark"] - comparison["Python"]
)
comparison.index.name = "Metric"

comparison.to_csv(
    BENCH_OUT / "23_test_metrics_comparison.csv"
)

print("TEST PERFORMANCE — PYTHON XGBOOST VS SPARKXGBCLASSIFIER")
display(comparison.style.format("{:.6f}"))

print("Threshold selected from OOF predictions; not adjusted on the test set.")
print("Brier score: lower is better; other performance metrics: higher is better.")
print("Decision threshold is a classification cut-off, not a performance metric.")

TEST PERFORMANCE — PYTHON XGBOOST VS SPARKXGBCLASSIFIER


engine,Python,Spark,Spark − Python
Metric,,,
Decision threshold,0.570808,0.570808,0.000000
ROC-AUC,0.860408,0.860408,0.000000
Average Precision (AP),0.737704,0.737704,0.000000
Accuracy,0.830521,0.830521,0.000000
Precision,0.654309,0.654309,0.000000
Recall,0.669765,0.669765,0.000000
F1-score,0.661947,0.661947,0.000000
Specificity,0.883463,0.883463,0.000000
Brier score,0.141933,0.141933,0.000000


Threshold selected from OOF predictions; not adjusted on the test set.
Brier score: lower is better; other performance metrics: higher is better.
Decision threshold is a classification cut-off, not a performance metric.
